In [1]:
pip install kaggle


   ---------------- ----------------------- 2/5 [python-slugify]
   ------------------------ --------------- 3/5 [bleach]
   ------------------------ --------------- 3/5 [bleach]
   ------------------------ --------------- 3/5 [bleach]
   -------------------------------- ------- 4/5 [kaggle]
   ---------------------------------------- 5/5 [kaggle]

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [3]:
import pandas as pd


In [6]:
movies = pd.read_csv(r"C:\Rahul KM Ghosh_PGD202564310\Innomatics Research Lab\GEN_AI\NLP_2_Assignment_GEN\IMDB Dataset.csv")

In [7]:
movies

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [11]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [14]:
print(movies.shape)
print(movies.head())
print(movies.info())

(50000, 2)
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB
None


In [15]:
movies["sentiment"].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [27]:
# movies.isnull().sum()
movies.duplicated().sum()

np.int64(418)

In [26]:
movies.drop_duplicates()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [28]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

In [40]:
# import nltk

# # Correct downloads
# nltk.download('stopwords')  # <--- note the 's' at the end
# nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to C:\Users\Tech Lab
[nltk_data]     1\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [33]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

In [46]:
def preprocess_text_simple(text):
    text = text.lower()  # lowercase
    text = re.sub(r"http\S+", "", text)  # remove URLs
    text = re.sub(r"[^a-zA-Z]", " ", text)  # remove punctuation/numbers
    tokens = text.split()  # simple tokenization
    tokens = [word for word in tokens if word not in stop_words]  # remove stopwords
    tokens = [stemmer.stem(word) for word in tokens]  # stemming
    return " ".join(tokens)

# APPLY THIS FUNCTION — notice the name
movies['clean_text'] = movies['review'].apply(preprocess_text_simple)

In [47]:
movies[['review', 'clean_text']].head()

,review,clean_text
0,One of the other reviewers has mentioned that ...,one review mention watch oz episod hook right ...
1,A wonderful little production. <br /><br />The...,wonder littl product br br film techniqu unass...
2,I thought this was a wonderful way to spend ti...,thought wonder way spend time hot summer weeke...
3,Basically there's a family where a little boy ...,basic famili littl boy jake think zombi closet...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter mattei love time money visual stun film...


In [48]:
movies.columns

Index(['review', 'sentiment', 'clean_text'], dtype='object')

In [49]:
from sklearn.feature_extraction.text import CountVectorizer

# Initialize BoW vectorizer
bow_vectorizer = CountVectorizer(max_features=5000)  # limit vocab to top 5000 words
X_bow = bow_vectorizer.fit_transform(movies['clean_text'])

print("BoW shape:", X_bow.shape)  # (50000, 5000)

BoW shape: (50000, 5000)


In [52]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf_vectorizer.fit_transform(movies['clean_text'])

print("TF-IDF shape:", X_tfidf.shape)  

TF-IDF shape: (50000, 5000)


In [51]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(movies['sentiment'])  # positive=1, negative=0
print(y[:10])

[1 1 1 0 1 1 1 0 0 1]


In [56]:
from sklearn.model_selection import train_test_split
X = X_tfidf
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.2, 
                                                    random_state=42)
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

Train shape: (40000, 5000) Test shape: (10000, 5000)


In [57]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [58]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [62]:
y_pred_lr = lr_model.predict(X_test)

In [63]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred_lr)
print("Logistic Regression Accuracy:", accuracy)

Logistic Regression Accuracy: 0.886
